In [ ]:
!pip install diffusers transformers accelerate -q

# Diffusion Models — Introduction

Diffusion models power **Stable Diffusion**, **DALL-E 3**, and **Midjourney**.
They have largely replaced GANs as the dominant generative architecture since 2022.

**By the end of this notebook you will:**
- Understand the forward and reverse diffusion processes
- Visualise how noise is added step by step
- Use HuggingFace `diffusers` to generate images with a pretrained DDPM
- Run an optional text-to-image demo with Stable Diffusion


## 1. The Core Idea

**Forward process** (no learning, fixed): destroy data by adding Gaussian noise over T steps.

**Reverse process** (learned): train a network to undo one denoising step at a time.

```
FORWARD:  x0 (clean) -> x1 -> x2 -> ... -> xT (pure noise)
REVERSE:  xT (noise) -> ... -> x1 -> x0 (generated image)
```

**Why better than GANs?**
- No mode collapse — stable regression objective (predict the noise)
- Higher quality and diversity
- Scales well with model size and compute


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tf = transforms.Compose([transforms.ToTensor()])
mnist = datasets.MNIST('./data', train=False, download=True, transform=tf)
clean_img = mnist[0][0]  # shape [1, 28, 28]


## 2. Forward Diffusion — Noise Schedule Visualisation

The cosine schedule adds noise slowly at first and faster later, preserving more structure early in the diffusion.

In [ ]:
def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    x = torch.linspace(0, T, steps)
    alphas_cumprod = torch.cos(((x / T) + s) / (1 + s) * np.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clamp(betas, 0, 0.999)

T = 1000
betas = cosine_beta_schedule(T)
alphas = 1 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

def q_sample(x0, t):
    sqrt_alpha = alphas_cumprod[t].sqrt()
    sqrt_one_minus = (1 - alphas_cumprod[t]).sqrt()
    noise = torch.randn_like(x0)
    return sqrt_alpha * x0 + sqrt_one_minus * noise, noise

timesteps = [0, 50, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(timesteps), figsize=(16, 3))
fig.suptitle('Forward Diffusion: Clean Image to Pure Noise', fontsize=13, fontweight='bold')
for ax, t in zip(axes, timesteps):
    noisy, _ = q_sample(clean_img, t)
    ax.imshow(noisy.squeeze().numpy(), cmap='gray', vmin=-1, vmax=1)
    ax.set_title(f't = {t}', fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(betas.numpy()); axes[0].set(title='beta noise per step', xlabel='Timestep t')
axes[1].plot(alphas_cumprod.numpy(), color='orange')
axes[1].set(title='alpha_bar signal remaining', xlabel='Timestep t')
for ax in axes: ax.grid(True, alpha=0.3)
plt.suptitle('Cosine Noise Schedule', fontsize=12); plt.tight_layout(); plt.show()
print('At t=0: alpha_bar=1 (full signal). At t=T: alpha_bar=0 (pure noise).')


## 3. The Denoising U-Net

The reverse process is learned by a **U-Net** that predicts the noise `epsilon` at each step:

```
Loss = ||epsilon - epsilon_theta(x_t, t)||^2
```

The network takes:
- The noisy image `x_t`
- The timestep `t` (as a sinusoidal embedding)

And predicts the noise that was added.

For **text-to-image**: the text prompt is encoded by CLIP and injected into the U-Net via cross-attention.

In [ ]:
from diffusers import DDPMPipeline
import torch

print('Loading pretrained DDPM (CelebA-HQ unconditional face generation)...')
pipeline = DDPMPipeline.from_pretrained(
    'google/ddpm-celebahq-256',
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
).to(device)

torch.manual_seed(42)
with torch.no_grad():
    output = pipeline(batch_size=4, num_inference_steps=50)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('DDPM Generated Faces (50 denoising steps)', fontsize=12)
for ax, img in zip(axes, output.images):
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()
print('These faces do not exist — 100% AI generated!')


## 4. DDIM — Faster Sampling

DDPM requires 1000 steps at inference. **DDIM** achieves equivalent quality in 20-50 steps by taking deterministic, larger steps.

| Sampler | Steps | Deterministic |
|---------|-------|---------------|
| DDPM | 1000 | No |
| DDIM | 20-50 | Yes |
| DPM-Solver | 10-20 | Yes |

In [ ]:
from diffusers import DDIMScheduler
import time

pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)

torch.manual_seed(42)
t0 = time.time()
with torch.no_grad():
    output_ddim = pipeline(batch_size=4, num_inference_steps=20)
elapsed = time.time() - t0

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle(f'DDIM 20 steps (took {elapsed:.1f}s)', fontsize=12)
for ax, img in zip(axes, output_ddim.images):
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'DDIM with 20 steps: {elapsed:.1f}s  (DDPM 1000 steps would take much longer)')


## 5. Text-to-Image with Stable Diffusion (Optional — A100 recommended)

Stable Diffusion = diffusion model in a compressed **latent space** (4x smaller) with CLIP text conditioning.

In [ ]:
# Optional: Stable Diffusion text-to-image (requires ~8GB VRAM, A100 recommended)
# Uncomment all lines below to run:
# from diffusers import StableDiffusionPipeline
# sd = StableDiffusionPipeline.from_pretrained(
#     'runwayml/stable-diffusion-v1-5',
#     torch_dtype=torch.float16, safety_checker=None).to('cuda')
# prompts = [
#     'a golden retriever in a field of sunflowers, photorealistic',
#     'a futuristic city at night, neon lights, cyberpunk style',
# ]
# images = sd(prompts, num_inference_steps=30, height=512, width=512).images
# fig, axes = plt.subplots(1, 2, figsize=(12, 6))
# for ax, img, p in zip(axes, images, prompts):
#     ax.imshow(img); ax.set_title(p[:50], fontsize=8); ax.axis('off')
# plt.tight_layout(); plt.show()
print('Stable Diffusion cell is commented out. Uncomment and run on A100 Colab.')


## 6. Exercises

1. **Noise schedule**: Implement a linear beta schedule (`betas = torch.linspace(1e-4, 0.02, T)`) and compare the forward process visualisation with cosine. Which preserves structure longer?
2. **Inference steps**: Try `num_inference_steps=10`, `25`, `50`, `100` with DDIM. At what step count does quality plateau?
3. **Different seeds**: Run the DDPM pipeline with 5 different seeds. How much do outputs vary? What does this reveal about the model's diversity?
